# Card sorting sobre PRs aceptados despues de retrabajo

## Pregunta de investigacion

**¿Qué tipos de problemas hacen que los pull requests generados por agentes de IA no sean aceptados inmediatamente y requieran un retrabajo antes de integrarse?**

Este notebook documenta el flujo que transforma el universo AIDev en una muestra revisable mediante card sorting abierto. El foco es `merged_after_rework`: PRs que fueron mergeados, pero solo despues de commits adicionales y comentarios humanos observables. Por eso hablamos de aceptacion no inmediata, no de rechazo definitivo.


## Problema, motivacion y consecuencias

Medir solo si un PR fue mergeado no explica que ocurrio durante la revision. Un PR puede terminar aceptado y aun asi requerir correcciones, aclaraciones o ajustes sustantivos antes del merge.

El objetivo es caracterizar esa zona intermedia: contribuciones de agentes de IA que necesitaron intervencion humana y retrabajo antes de integrarse. La evidencia usada para categorizar debe responder directamente por que el PR no fue aceptado inmediatamente.


## Enfoque metodologico segun Zimmermann

Adaptamos `docs/card-sorting.pdf`, que organiza el card sorting en tres fases: **Preparation**, **Execution** y **Analysis**. Usamos card sorting abierto: las categorias emergen desde las tarjetas y luego se consolidan en una taxonomia.

### 1) Preparation

Construir la poblacion operacional, declarar criterios de inclusion/exclusion, medir perdidas, estratificar por agente y generar una tarjeta por PR con identificador, contexto y evidencia textual humana.

### 2) Execution

Clasificar manualmente las tarjetas en grupos con nombres descriptivos. Las tarjetas ambiguas o sin evidencia suficiente se separan para revision y no se fuerzan dentro de categorias tecnicas.

### 3) Analysis

Revisar consistencia, fusionar o dividir grupos, congelar una taxonomia jerarquica y cruzarla con metricas de agente, lenguaje, tipo de tarea, comentarios humanos y tiempo hasta aceptacion. Esta fase queda pendiente hasta completar la categorizacion manual.


## Carga del flujo reproducible

El notebook no implementa logica de extraccion, muestreo ni preparacion directamente. Importa funciones desde `exploration.aidev.notebook_flow`, que ejecuta el flujo en orden si falta algun output:

1. `sampling/population_filter.py`: aplica filtros poblacionales y escribe la poblacion operacional.
2. `sampling/stratified_sampler.py`: lee ese CSV y genera la muestra estratificada.
3. `preparation/rejection_cards.py`: lee la muestra y genera tarjetas con evidencia.

La celda siguiente inicializa rutas, carga artefactos y muestra conteos de control. Asi el notebook funciona como documento de analisis y mantiene la logica reusable en scripts importables.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def resolve_notebook_repo_root() -> Path:
    for start in [Path.cwd().resolve(), Path.cwd().resolve().parent]:
        for candidate in [start, *start.parents]:
            if (candidate / "exploration" / "aidev").exists():
                return candidate
    raise RuntimeError("No se encontro la raiz del repositorio; abre el notebook dentro del repo")


ROOT = resolve_notebook_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from exploration.aidev.notebook_flow import (
    build_agent_distribution,
    build_evidence_tables,
    build_funnel,
    build_outputs_flow,
    build_population_artifact_table,
    build_population_filter_table,
    build_preparation_flow,
    build_sampling_artifact_table,
    ensure_flow_outputs,
    find_repo_root,
    validate_flow,
)
from exploration.aidev.sampling.population_filter import POPULATION_MODE
from exploration.aidev.sampling.stratified_sampler import STRATA_FIELDS

ROOT = find_repo_root(ROOT)
artifacts = ensure_flow_outputs(ROOT)
population_summary = artifacts.population_summary
sampling_summary = artifacts.sampling_summary
preparation_summary = artifacts.preparation_summary
filter_counts = population_summary["population_filter_counts"]
population_df = artifacts.population_df
sample_df = artifacts.sample_df
cards_df = artifacts.cards_df
template_df = artifacts.template_df

{
    "population_mode": POPULATION_MODE,
    "strata_fields": STRATA_FIELDS,
    "population_rows": len(population_df),
    "sample_rows": len(sample_df),
    "card_rows": len(cards_df),
}


## Paso 0: filtros poblacionales antes de estratificar

Script: `exploration/aidev/sampling/population_filter.py`.

Antes de seleccionar la muestra, construimos la poblacion operacional completa. El criterio aplicado es:

```text
merged_at no nulo
commit_count > 1
human_comment_count > 0
population_case_type = merged_after_rework
```

Este paso produce dos artefactos independientes: un resumen JSON con metricas del embudo y un CSV con los `3.166` PRs elegibles antes de la estratificacion. Separarlo del muestreo permite auditar la poblacion base, revisar distribuciones y repetir la seleccion de muestra sin recalcular los filtros.


In [ ]:
build_population_filter_table(artifacts)


In [ ]:
build_population_artifact_table(artifacts)


In [ ]:
population_df[[
    "pr_id",
    "agent",
    "language",
    "task_type",
    "commit_count",
    "human_comment_count",
    "html_url",
]].head()


## Paso 1: estratificacion por agente

Script: `exploration/aidev/sampling/stratified_sampler.py`.

La muestra se toma desde `merged_after_rework_population.csv`, producido en el paso 0. Primero se define el tamano total de muestra. Para estimar el tamano requerido con correccion por poblacion finita usamos:

```text
n = (N * Z^2 * p * q) / (e^2 * (N - 1) + Z^2 * p * q)
```

Con `N = 3166`, `Z = 1.96`, `p = 0.5`, `q = 0.5` y `e = 0.05`, el resultado es `n = 342.69`, por lo que el tamano recomendado para error maximo de 5% es `343` PRs.

La muestra vigente usa `n = 300`. Es reproducible y util para la ronda actual, pero su error aproximado es `+/- 5.38%`. Si se exige `<= 5%`, debe ampliarse a `n = 343`.

Una vez definido `n`, asignamos cuotas proporcionales por agente:

```text
n_h = max(m, round(n * N_h / N))
```

Donde `N_h` es el tamano del estrato del agente, `N` es la poblacion operacional, `n` es el tamano total y `m` es el minimo por estrato cuando se aplica. Con `n = 300` y seed `20260510`, el sampler genera un CSV de muestra y un JSON de cuotas/distribuciones.


In [ ]:
build_sampling_artifact_table(artifacts)


In [ ]:
build_agent_distribution(artifacts)


In [ ]:
embudo = build_funnel(artifacts).copy()
for columna in ["retencion_vs_universo", "retencion_vs_paso_anterior"]:
    embudo[columna] = embudo[columna].map(lambda value: f"{value:.2%}")
embudo


## Paso 2: Preparation de tarjetas

En la fase **Preparation**, cada PR de la muestra se transforma en una tarjeta. La tarjeta conserva `card_id`, `pr_id`, contexto del PR, agente, tiempos de aceptacion y evidencia textual. El filtro `human_comment_count > 0` queda como guardia de calidad.

La preparacion prioriza evidencia en este orden: reviews con cambios solicitados, comentarios inline humanos, comentarios generales del PR, reviews/comentarios de bots, timeline y titulo/body como respaldo.


In [ ]:
build_preparation_flow(artifacts)


### Evidencia disponible tras Preparation

La tabla resume la fuente principal seleccionada para cada tarjeta y los estados de review observados. Esta evidencia es la base de la cita textual que debe sostener la categorizacion manual.


In [ ]:
evidence, review_states = build_evidence_tables(artifacts)
display(evidence)
display(review_states)


### Salidas para categorizacion manual

Las tarjetas y la plantilla manual son continuacion directa de Preparation. Todavia no constituyen resultados cualitativos finales; son los insumos que se revisan durante Execution.


In [ ]:
build_outputs_flow(artifacts)


### Tabla reducida para categorizar

Para sostener soundness, cada hoja manual debe conservar trazabilidad `card_id -> PR -> evidencia -> categoria`. La categoria solo es defendible si queda respaldada por una cita humana que responda la pregunta de investigacion. Si no existe evidencia suficiente, la tarjeta debe marcarse como ambigua o sin evidencia suficiente.

Los avances manuales por evaluador se guardan en `exploration/aidev/taxonomy/initial/` con el mismo esquema de columnas y los mismos `card_id`.


In [ ]:
columnas_categorizacion = pd.DataFrame([
    {"columna": "card_id", "uso_metodologico": "identificador unico para reconstruir la tarjeta"},
    {"columna": "pr_id", "uso_metodologico": "trazabilidad hacia el PR original"},
    {"columna": "agent", "uso_metodologico": "estrato/agente de origen"},
    {"columna": "html_url", "uso_metodologico": "enlace para revisar contexto si la cita no basta"},
    {"columna": "cita_textual_retrabajo", "uso_metodologico": "fragmento humano que sustenta la categoria"},
    {"columna": "evidence_source", "uso_metodologico": "fuente de la cita seleccionada"},
    {"columna": "evidence_created_at", "uso_metodologico": "fecha de la evidencia usada como cita"},
    {"columna": "merged_at", "uso_metodologico": "control temporal para distinguir evidencia pre/post merge"},
    {"columna": "categoria_retrabajo_pre_merge", "uso_metodologico": "categoria emergente asignada durante el card sorting"},
    {"columna": "justificacion_breve", "uso_metodologico": "explicacion breve de por que la cita responde la pregunta"},
])
columnas_categorizacion


In [ ]:
manual_categories_paths = {
    "Javier": ROOT / "exploration/aidev/taxonomy/initial/merged_after_rework_manual_categories_Javier.csv",
    "diego": ROOT / "exploration/aidev/taxonomy/initial/merged_after_rework_manual_categories_diego.csv",
}
manual_categories_index = pd.DataFrame([
    {
        "evaluador": evaluador,
        "ruta": str(path.relative_to(ROOT)).replace("\\", "/"),
        "existe": path.exists(),
        "filas": len(pd.read_csv(path)) if path.exists() else 0,
    }
    for evaluador, path in manual_categories_paths.items()
])
display(manual_categories_index)

columnas_vista = [
    "card_id",
    "pr_id",
    "agent",
    "html_url",
    "cita_textual_retrabajo",
    "categoria_retrabajo_pre_merge",
    "justificacion_breve",
]
manual_previews = []
for evaluador, path in manual_categories_paths.items():
    if not path.exists():
        continue
    tabla = pd.read_csv(path)
    manual_previews.append(
        tabla[columnas_vista].head(3).assign(evaluador=evaluador)
    )

pd.concat(manual_previews, ignore_index=True)[["evaluador", *columnas_vista]]


## Paso 3: Ejecucion y analisis

Este paso queda pendiente hasta completar la categorizacion manual. Siguiendo la tercera fase de Zimmermann, el analisis no consiste solo en contar etiquetas: primero hay que revisar consistencia interna, fusionar grupos equivalentes, dividir grupos demasiado amplios y congelar una taxonomia estable.

Cuando la taxonomia este completa, el analisis debe producir:

- categorias y subcategorias con definicion, inclusion, exclusion y ejemplos;
- tarjetas ambiguas o sin evidencia suficiente separadas explicitamente;
- frecuencias descriptivas por categoria, sin interpretarlas como importancia absoluta;
- cruces por agente, lenguaje, tipo de tarea, complejidad y tiempos de aceptacion;
- revision de desacuerdos o doble codificacion si participa mas de un evaluador.

Hasta entonces, las tablas manuales deben tratarse como insumo de sorting, no como resultado cualitativo final.


## Validaciones de consistencia

Estas validaciones hacen explicitos los supuestos automatizados del flujo: poblacion `merged_after_rework`, CSV poblacional pre-muestreo, estratificacion por agente, muestra de 300 PRs y una tarjeta final por PR.


In [ ]:
validate_flow(artifacts)


## Lectura metodologica y soundness

El flujo parte del universo completo de PRs, filtra antes del muestreo los casos que no muestran retrabajo observable y conserva una muestra estratificada por agente. La interpretacion cualitativa debe enfocarse en motivos de retrabajo antes del merge, no en rechazo definitivo.

Para aplicar soundness, cada etiqueta manual debe cumplir cinco condiciones: (1) estar respaldada por una cita textual humana cuando exista evidencia; (2) responder directamente que problema impidio aceptacion inmediata; (3) distinguir retrabajo pre-merge de rechazo definitivo; (4) conservar trazabilidad `card_id -> PR -> evidencia -> categoria`; y (5) no tratar la frecuencia de tarjetas como importancia absoluta, siguiendo la cautela metodologica de Zimmermann sobre cuantificar datos cualitativos.

La presentacion debe cubrir: problema y pregunta; dataset AIDev; filtros poblacionales con perdidas porcentuales; CSV poblacional de 3.166 PRs; formula de tamano muestral; formula de cuotas por agente; muestra vigente `n = 300` y mejora recomendada `n = 343`; Preparation de tarjetas; tabla reducida con cita textual; soundness; y resultados esperados de taxonomia mas metricas de esfuerzo/tiempo.
